# 🤖 Week 5: AI Email Assistant with LangChain & Memory

## 📚 What You'll Learn:
- How to set up LangChain with OpenAI
- Understanding the required libraries for email automation
- Initializing the LLM with appropriate parameters
- Building prompt templates and chains
- Implementing conversation memory for multi-turn email threads
- Processing real emails from Gmail/Outlook

---

## 🎯 Quick Function Reference Guide

| Function | Purpose | Use Case | Example |
|----------|---------|----------|---------|
| **`generate_email_reply()`** | Basic email reply generation | Simple, one-off email responses | `reply = generate_email_reply(email_data)` |
| **`generate_personalized_reply()`** | Reply with custom tone | Adjust communication style | `reply = generate_personalized_reply(email_data, tone="friendly")` |
| **`generate_reply_with_memory()`** | Reply with thread context | Multi-turn conversations | `reply = generate_reply_with_memory(email_data, thread_id="ticket-123")` |
| **`get_session_history()`** | Access thread memory | Check conversation history | `history = get_session_history("customer-abc")` |
| **`parse_eml_file()`** | Parse single .eml file | Process one Gmail export | `email = parse_eml_file("message.eml")` |
| **`parse_eml_directory()`** | Parse all .eml files in folder | Batch import Gmail folder | `emails = parse_eml_directory("./emails/")` |
| **`save_replies_to_csv()`** | Export replies to CSV | Store for audit/review | `df = save_replies_to_csv(replies)` |

---

## 🚀 Three Ways to Use This Notebook

### **Option 1: Quick Email Reply** (2 minutes)
```python
email = {
    "id": 1,
    "sender": "customer@company.com",
    "sender_name": "John Smith",
    "subject": "Question about pricing",
    "body": "Hi, can you tell me your pricing?",
    "email_type": "client",
    "priority": "high"
}

reply = generate_email_reply(email, your_name="Sarah")
print(reply['reply_body'])
```

### **Option 2: Batch Process Multiple Emails** (5 minutes)
```python
# We've provided 5 sample emails
for email in sample_emails:
    reply = generate_email_reply(email)
    print(f"Processed: {email['sender_name']}")
```

### **Option 3: Complete Production Workflow** (20 minutes)
```python
# Export emails from Gmail → Parse → Generate replies → Save results
emails = parse_eml_directory("./my_gmail_exports/")
for email in emails:
    reply = generate_reply_with_memory(email, thread_id=f"thread-{email['id']}")
```

---

## 📂 Notebook Structure

1. **Setup & Imports** - Install and configure LangChain + OpenAI
2. **Sample Data** - Create realistic email dataset
3. **Basic Replies** - Simple email response generation
4. **Batch Processing** - Process multiple emails
5. **Tone Customization** - Adjust communication styles
6. **Memory System** - Thread-aware multi-turn conversations
7. **Thread Isolation Demo** - Show how different threads stay separate
8. **Bonus: Real Emails** - Parse Gmail/Outlook .eml files

---

## 💡 Key Concepts Explained

**Chain (Pipeline):**
- Input (Email) → PromptTemplate → LLM → Output (Reply)
- We use LCEL syntax: `prompt | llm`

**Memory (Context):**
- First email in thread: Creates new memory
- Second email: AI remembers first email and references it
- Each thread has independent memory

**Thread ID:**
- Unique identifier for conversation
- Example: "customer-support" or "sales-proposal"
- Different threads = different memories

---

In [1]:
# TODO: Import all required libraries
# HINT: You'll need:
# - os, json, pandas, dotenv, datetime
# - ChatOpenAI from langchain_openai
# - PromptTemplate, ChatPromptTemplate, MessagesPlaceholder from langchain_core.prompts
# - RunnableWithMessageHistory from langchain_core.runnables.history
# - BaseChatMessageHistory, InMemoryChatMessageHistory from langchain_core.chat_history

import os
import json
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime

# LangChain imports (LangChain 1.0.8 + langchain-community for memory)
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory

# Load environment variables
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Verify API key
if not api_key:
    raise ValueError("❌ ERROR: Please set OPENAI_API_KEY in your .env file")

print("✅ All libraries imported successfully")
print("✅ OpenAI API Key found")
print("\n🎯 Ready to build your AI Email Assistant!\n")

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries imported successfully
✅ OpenAI API Key found

🎯 Ready to build your AI Email Assistant!



### Initialize the LLM

**Key Parameters:**
- `model`: We use `gpt-4o-mini` for fast, cost-effective email generation
- `temperature`: 0.7 balances creativity with consistency
- `max_tokens`: 500 is sufficient for most professional emails

16k tokens - 20 emails + user question + AI Response = Context Window

In [2]:
# TODO: Initialize ChatOpenAI
# HINT: Use ChatOpenAI() with api_key, model, temperature, and max_tokens parameters

# Step 1: Create the LLM instance
llm = None  # Replace with ChatOpenAI(...)

llm = ChatOpenAI(
    api_key = api_key,
    model = "gpt-4o-mini",
    temperature = 0.7,
    max_tokens = 500
)

# Step 2: Print confirmation
print("✅ OpenAI LLM initialized successfully")
print(f"   Model: gpt-4o-mini")
print(f"   Temperature: 0.7 (balanced creativity)")
print(f"   Max Tokens: 500")

✅ OpenAI LLM initialized successfully
   Model: gpt-4o-mini
   Temperature: 0.7 (balanced creativity)
   Max Tokens: 500


---

## Part 2: Create Sample Email Dataset

### 📚 What You'll Learn:
- How to structure email data for processing (Gmail-compatible format)
- Categorizing emails by type and priority
- Using pandas for data organization

In [3]:
# TODO: Create sample email dataset
# HINT: Create a list of dictionaries, each with:
# - id, sender, sender_name, subject, body, email_type, priority

# Step 1: Create list with 5 email dictionaries
# Create sample email dataset with Indian names and detailed bodies
sample_emails = [
    {
        "id": 1,
        "sender": "rajesh.kumar@techsolutions.in",
        "sender_name": "Rajesh Kumar",
        "subject": "Project Update - Q4 Goals and Team Alignment",
        "body": "Hi team, I hope this email finds you well. As we approach the end of Q3, I wanted to reach out to discuss our Q4 goals and how we can better align our efforts across all departments. We've made significant progress on the cloud migration project, but there are still some challenges with the timeline that need to be addressed. I'd like to schedule a meeting next week to review our current status, identify any bottlenecks, and ensure everyone is on the same page regarding priorities. Please let me know your availability for Tuesday or Wednesday afternoon. Looking forward to a productive discussion.",
        "email_type": "business",
        "priority": "high"
    },
    {
        "id": 2,
        "sender": "priya.sharma@clientcorp.com",
        "sender_name": "Priya Sharma",
        "subject": "Feedback on Proposal - Pricing and Implementation Clarification",
        "body": "Dear team, Thank you so much for sending over the detailed proposal for the enterprise software solution. We've reviewed it thoroughly with our stakeholders, and overall, we're very impressed with the features and timeline you've outlined. However, we do have some questions regarding the pricing model, particularly around the tiered structure and what's included in each tier. Could you please clarify the differences between the Standard and Premium packages? Additionally, we'd like to understand the implementation timeline better - specifically, how the phases would work for our organization size of 500+ employees. We're also interested in knowing if there's any flexibility in customizing certain modules to fit our specific workflow requirements. Would it be possible to schedule a call this week to discuss these points in detail?",
        "email_type": "client",
        "priority": "high"
    },
    {
        "id": 3,
        "sender": "hr@innovatetech.in",
        "sender_name": "Neha Patel - HR Department",
        "subject": "Annual Performance Review Schedule - Action Required",
        "body": "Dear Employee, This is to inform you that your annual performance review has been scheduled for next week, specifically on Thursday, October 12th at 2:00 PM. As part of the preparation process, please complete your self-assessment form and submit it through our HR portal by Monday, October 9th. The self-assessment should include your key accomplishments over the past year, areas where you feel you've grown professionally, challenges you've faced, and your goals for the upcoming year. Additionally, please prepare any documentation that supports your achievements, such as project completion reports, client feedback, or metrics that demonstrate your contributions to the team. Your manager will also be completing a separate evaluation, and both assessments will be discussed during the review meeting. If you have any questions about the process or need assistance accessing the portal, please don't hesitate to reach out to the HR team.",
        "email_type": "hr",
        "priority": "medium"
    },
    {
        "id": 4,
        "sender": "arjun.verma@globalpartners.com",
        "sender_name": "Arjun Verma",
        "subject": "Strategic Partnership Opportunity - AI Solutions Integration",
        "body": "Hello, I hope you're doing well. My name is Arjun Verma, and I'm the Business Development Manager at Global Partners. I've been following your company's impressive work in the AI and machine learning space, particularly your recent launch of the automated customer service platform. We specialize in providing enterprise integration solutions and have a strong presence in the retail and e-commerce sectors across India and Southeast Asia. I believe there could be a fantastic synergy between our companies - specifically, integrating your AI platform with our existing client base could create tremendous value for both organizations. We have over 200 enterprise clients who are actively looking for advanced AI solutions, and your technology seems like a perfect fit. I'd love to explore this opportunity further and discuss how we might structure a mutually beneficial partnership. Would you be available for a brief introductory call sometime next week? I'm flexible with timing and happy to work around your schedule.",
        "email_type": "business",
        "priority": "low"
    },
    {
        "id": 5,
        "sender": "events@indiatechsummit.com",
        "sender_name": "India Tech Summit 2024",
        "subject": "Early Bird Registration Extended - Save 30% on Tech Summit Passes",
        "body": "Greetings Tech Enthusiast! We're excited to announce that due to popular demand, we've extended our Early Bird registration period for the India Tech Summit 2024! You now have until the end of this month to secure your spot at India's largest technology conference and save 30% on all ticket types. The summit will take place from December 15-17, 2024, at the Mumbai Convention Center and will feature over 100 speakers from leading tech companies including Google, Microsoft, Amazon, and top Indian startups. This year's agenda includes keynotes on AI/ML, Cloud Computing, Blockchain, Cybersecurity, and the Future of Work. You'll also have access to hands-on workshops, networking sessions with industry leaders, and an exclusive startup exhibition showcasing the most innovative products in the market. Don't miss this opportunity to learn from the best, expand your professional network, and stay ahead of the technology curve. Register now using code EARLYBIRD30 to claim your discount. Visit our website for the full agenda and speaker lineup. We look forward to seeing you there!",
        "email_type": "notification",
        "priority": "low"
    }
]

# Step 2: Convert to DataFrame for visualization
emails_df = pd.DataFrame(sample_emails)  # TODO: Use pd.DataFrame(sample_emails)

# Step 3: Display the dataset
print("📧 Sample Email Dataset Created:")
print("="* 100)
print(emails_df[["id", 'sender_name', 'subject', 'email_type']].to_string())                 # TODO: Print the DataFrame with selected columns
print("="* 100)
print(f"\n✅ Total emails loaded: {len(emails_df)}")

📧 Sample Email Dataset Created:
   id                 sender_name                                                            subject    email_type
0   1                Rajesh Kumar                       Project Update - Q4 Goals and Team Alignment      business
1   2                Priya Sharma    Feedback on Proposal - Pricing and Implementation Clarification        client
2   3  Neha Patel - HR Department               Annual Performance Review Schedule - Action Required            hr
3   4                 Arjun Verma       Strategic Partnership Opportunity - AI Solutions Integration      business
4   5      India Tech Summit 2024  Early Bird Registration Extended - Save 30% on Tech Summit Passes  notification

✅ Total emails loaded: 5


---

## Part 3: Build Email Reply Chain

### 📚 What You'll Learn:
- How to design effective prompts for email generation
- Using LangChain's LCEL (LangChain Expression Language)
- Creating reusable chains for email processing

1. Context is king: We provode the sender info, email type
2. Clear Instructions
3. Personnalization : user's name, users title, company
4. LCEL Syntax

In [ ]:
# 🔗 STEP 4: Build Email Reply Chain\n# ====================================\n# This is the core logic that instructs the LLM how to generate professional email replies\n\n# Define the prompt template with placeholders for dynamic content\nemail_reply_template = \"\"\"You are a professional email assistant. Generate a personalized, \nprofessional email reply based on the following context.\n\nSender Name: {sender_name}\nEmail Type: {email_type}\nPriority Level: {priority}\nOriginal Email Subject: {subject}\nOriginal Email Body: {body}\nYour Name: {your_name}\nYour Title: {your_title}\nCompany: {company}\n\nGenerate a professional, personalized reply that:\n1. Addresses the sender by name\n2. Acknowledge their emails properly\n3. Provide a thoughtful response that is relevant to the email type\n4. Maintain a professional tone\n5. Include a clear call-to-action or next steps\n\nReply Email: \"\"\"\n\n# Create a PromptTemplate object with the template and input variables\n# Variables in {curly_braces} will be replaced with actual values at runtime\nprompt = PromptTemplate(\n    input_variables=[\"sender_name\", \"email_type\", \"priority\", \"subject\", \"body\", \"your_name\", \"your_title\", \"company\"],\n    template=email_reply_template\n)\n\n# Create a chain using LCEL (LangChain Expression Language)\n# The pipe (|) operator chains components: Prompt → LLM\n# This means: Take the prompt template, fill in variables, then pass to LLM\nemail_chain = prompt | llm\n\n# Print confirmation and explanation\nprint(\"✅ Email reply generation chain created successfully\")\nprint(\"\\n🔗 Chain Structure: PromptTemplate | ChatOpenAI\")\nprint(\"   └─ This is LCEL (LangChain Expression Language) syntax\")\nprint(\"   └─ It passes the formatted prompt to the LLM for processing\")\nprint(\"\\n💡 How it works:\")\nprint(\"   1. PromptTemplate receives variable values\")\nprint(\"   2. Variables are substituted into the template\")\nprint(\"   3. Formatted prompt is sent to OpenAI API\")\nprint(\"   4. AI generates response based on instructions\")\nprint(\"   5. Response is returned as ChatMessage object\")", "oldString": "# TODO: Create email reply prompt template\n# HINT: Define a string template with variables in {curly_braces}\n\n# Step 1: Define the prompt template string\nemail_reply_template = \"\"\"You are a professional email assistant. Generate a personalized, \nprofessional email reply based on the following context.\n\nSender Name: {sender_name}\nEmail Type: {email_type}\nPriority Level: {priority}\nOriginal Email Subject: {subject}\nOriginal Email Body: {body}\nYour Name: {your_name}\nYour Title: {your_title}\nCompany: {company}\n\nGenerate a professional, personalized reply that:\n1. Addresses the sender by name\n2. Acknowledge their emails properly\n3. Provide a thoughtful response that is relevent to the email type\n4. Maintain a professional tone\n5. Include the clear call-to-action or next steps\n\nReply Email: \"\"\"\n\n# Step 2: Create PromptTemplate object\nprompt = PromptTemplate(\n    input_variables = [\"sender_name\", \"email_type\", \"priority\",\"subject\", \"body\", \"your_name\", \"your_title\", \"company\"],\n    template = email_reply_template\n)\n\n# Step 3: Create chain using LCEL (pipe operator |)\nemail_chain = prompt | llm\n\nprint(\"✅ Email reply generation chain created successfully\")\nprint(\"\\n🔗 Chain Structure: PromptTemplate | ChatOpenAI\")\nprint(\"   └─ This is LCEL (LangChain Expression Language) syntax\")"}}]
</invoke>

✅ Email reply generation chain created successfully

🔗 Chain Structure: PromptTemplate | ChatOpenAI
   └─ This is LCEL (LangChain Expression Language) syntax


### Create the Reply Generation Function

This function wraps our chain in error handling and data formatting.

In [7]:
email_data =     {
        "id": 4,
        "sender": "arjun.verma@globalpartners.com",
        "sender_name": "Arjun Verma",
        "subject": "Strategic Partnership Opportunity - AI Solutions Integration",
        "body": "Hello, I hope you're doing well. My name is Arjun Verma, and I'm the Business Development Manager at Global Partners. I've been following your company's impressive work in the AI and machine learning space, particularly your recent launch of the automated customer service platform. We specialize in providing enterprise integration solutions and have a strong presence in the retail and e-commerce sectors across India and Southeast Asia. I believe there could be a fantastic synergy between our companies - specifically, integrating your AI platform with our existing client base could create tremendous value for both organizations. We have over 200 enterprise clients who are actively looking for advanced AI solutions, and your technology seems like a perfect fit. I'd love to explore this opportunity further and discuss how we might structure a mutually beneficial partnership. Would you be available for a brief introductory call sometime next week? I'm flexible with timing and happy to work around your schedule.",
        "email_type": "business",
        "priority": "low"
    }

In [8]:
email_data.get("sender_name", "")

'Arjun Verma'

In [9]:
email_data['sender_name']

'Arjun Verma'

In [ ]:
Sender Name: {sender_name}
Email Type: {email_type}
Priority Level: {priority}
Original Email Subject: {subject}
Original Email Body: {body}
Your Name: {your_name}
Your Title: {your_title}
Company: {company}

In [ ]:
result.content if hasattr(result, 'content') else str(result)

if "result" has a .content attribute, use that as the reply; otherwise convert that result to a string and 

In [10]:
# TODO: Create generate_email_reply function
# HINT: Function should invoke the chain and return a structured dict

def generate_email_reply(email_data, your_name="Sarvesh", 
                        your_title="Delivery Manager", company="Learn With Sarvesh"):
    """
    Generate a personalized email reply using LLM
    
    Args:
        email_data (dict): Email information (sender, subject, body, etc.)
        your_name (str): Your name for the signature
        your_title (str): Your job title
        company (str): Your company name
    
    Returns:
        dict: Generated reply with metadata
    """
    try:
        # Step 1: Invoke the chain with email data
        result = email_chain.invoke({
            "sender_name": email_data.get("sender_name", ""),
            "email_type": email_data.get("email_type", ""),
            "priority": email_data.get("priority", ""),
            "subject": email_data.get("subject", ""),
            "body": email_data.get("body", ""),
            "your_name": your_name,
            "your_title": your_title,
            "company": company
        })
        
        # Step 2: Extract content from response
        reply = result.content if hasattr(result, 'content') else str(result)
        
        # Step 3: Return structured response
        # TODO: Add all return fields
        # - original_email_id, from, to, subject, reply_body, generated_at, status
        return {
            'original_email_id': email_data.get("id"),
            "from": your_name,
            "to": email_data.get("sender"),
            "subject": f"Re: {email_data.get('subject')}",
            "reply_body": reply.strip(),
            "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "status": "generated"
        }
    
    except Exception as e:
        # TODO: Handle errors gracefully
        return {"original_email_id": email_data.get("id"), 
                "error": str(e), 
                "status": "failed"}

print("✅ Email reply generation function created successfully")
print("\n📝 Function signature: generate_email_reply(email_data, your_name, your_title, company)")

✅ Email reply generation function created successfully

📝 Function signature: generate_email_reply(email_data, your_name, your_title, company)


In [ ]:
email_data =     {
        "id": 4,
        "sender": "arjun.verma@globalpartners.com",
        "sender_name": "Arjun Verma",
        "subject": "Strategic Partnership Opportunity - AI Solutions Integration",
        "body": "Hello, I hope you're doing well. My name is Arjun Verma, and I'm the Business Development Manager at Global Partners. I've been following your company's impressive work in the AI and machine learning space, particularly your recent launch of the automated customer service platform. We specialize in providing enterprise integration solutions and have a strong presence in the retail and e-commerce sectors across India and Southeast Asia. I believe there could be a fantastic synergy between our companies - specifically, integrating your AI platform with our existing client base could create tremendous value for both organizations. We have over 200 enterprise clients who are actively looking for advanced AI solutions, and your technology seems like a perfect fit. I'd love to explore this opportunity further and discuss how we might structure a mutually beneficial partnership. Would you be available for a brief introductory call sometime next week? I'm flexible with timing and happy to work around your schedule.",
        "email_type": "business",
        "priority": "low"
    }

---

## Part 4: Generate Replies for All Emails

### 📚 What You'll Learn:
- Batch processing multiple emails
- Handling successes and failures
- Displaying results in a user-friendly format

1. Rate limiting -> to avoid API throttling 
2. Async proccessing for speed
3. Retry logic for failed responses

### Rate Limiting:
Every API provider (Openai) sets a rule that only N requests can be handled in T seconds.

- N = 10 requests
- T = 10 seconds

API Throttling -> API will slow you down, rejects the requests , Temporarlity block you -> API throttling

1. Control infra costs
2. Maintain system stabity

In [ ]:
email_data =     {
        "id": 4,
        "sender": "arjun.verma@globalpartners.com",
        "sender_name": "Arjun Verma",
        "subject": "Strategic Partnership Opportunity - AI Solutions Integration",
        "body": "Hello, I hope you're doing well. My name is Arjun Verma, and I'm the Business Development Manager at Global Partners. I've been following your company's impressive work in the AI and machine learning space, particularly your recent launch of the automated customer service platform. We specialize in providing enterprise integration solutions and have a strong presence in the retail and e-commerce sectors across India and Southeast Asia. I believe there could be a fantastic synergy between our companies - specifically, integrating your AI platform with our existing client base could create tremendous value for both organizations. We have over 200 enterprise clients who are actively looking for advanced AI solutions, and your technology seems like a perfect fit. I'd love to explore this opportunity further and discuss how we might structure a mutually beneficial partnership. Would you be available for a brief introductory call sometime next week? I'm flexible with timing and happy to work around your schedule.",
        "email_type": "business",
        "priority": "low"
    }

Expotential Backoff: 
1. if error or time limits (min(3mins) ) * N times it tried and failed - Backoff -> Dont break the system arch

In [12]:
# TODO: Process all emails and generate replies
# HINT: Loop through sample_emails, call generate_email_reply() for each

print("🚀 Generating replies for all emails...\n")
print("=" * 80)

generated_replies = []

# Step 1: Loop through all emails
for idx, email in enumerate(sample_emails, 1):
    # TODO: Print email info
    print(f"\n Email {idx}/{len(sample_emails)}: {email['subject']}")
    print(f" From: {email['sender_name']}")
    print(f" Type: {email['email_type'].upper()} | Priority: {email['priority'].upper()}")
    print("-"*80)
    
    # Step 2: Generate reply
    reply = generate_email_reply(email)
    
    # TODO: Append to generated_replies
    generated_replies.append(reply)

    # Step 3: Show status and preview
    if reply.get("status") == "generated":
        print(f"Reply generated successfully")
        print(f" Subject: {reply['subject']}")
        print(f"\n Preview (first 3 lines):")
        print("-"*50)
        preview_lines = reply['reply_body'].split('\n')[:3]

        for line in preview_lines:
            print(f" {line}")
        print("   " + "-" * 70)

    # TODO: Print success/failure status
    # TODO: Print preview of first 3 lines if successful

# Step 4: Print summary statistics
print("\n" + "=" * 80)
print(f"\n✅ Processing Complete!")
print(f" Total: {len(generated_replies)} email processed")
print(f" Success: {sum(1 for r in generated_replies if r.get('status') == 'generated')}")
print(f" Failed: {sum(1 for r in generated_replies if r.get('status') == 'failed')}")
# TODO: Print total, success, and failed counts

🚀 Generating replies for all emails...


 Email 1/5: Project Update - Q4 Goals and Team Alignment
 From: Rajesh Kumar
 Type: BUSINESS | Priority: HIGH
--------------------------------------------------------------------------------
Reply generated successfully
 Subject: Re: Project Update - Q4 Goals and Team Alignment

 Preview (first 3 lines):
--------------------------------------------------
 Subject: Re: Project Update - Q4 Goals and Team Alignment
 
 Hi Rajesh,
   ----------------------------------------------------------------------

 Email 2/5: Feedback on Proposal - Pricing and Implementation Clarification
 From: Priya Sharma
 Type: CLIENT | Priority: HIGH
--------------------------------------------------------------------------------
Reply generated successfully
 Subject: Re: Feedback on Proposal - Pricing and Implementation Clarification

 Preview (first 3 lines):
--------------------------------------------------
 Subject: Re: Feedback on Proposal - Pricing and Implementati

### 🔍 View Full Generated Replies

Let's examine the complete replies to see the quality and personalization.

In [16]:
# Display detailed view of generated replies
print("\n📨 DETAILED VIEW OF GENERATED REPLIES\n")
print("=" * 100)

for i, reply in enumerate(generated_replies, 1):
    if reply.get("status") == "generated":
        original_email = sample_emails[i-1]
        
        print(f"\n{'REPLY ' + str(i):^100}")
        print("=" * 100)
        
        # Original email
        print(f"\n📬 ORIGINAL EMAIL:")
        print(f"   From: {original_email['sender_name']} <{original_email['sender']}>")
        print(f"   Subject: {original_email['subject']}")
        print(f"   Type: {original_email['email_type'].upper()} | Priority: {original_email['priority'].upper()}")
        print(f"\n   Message:\n   {original_email['body']}")
        
        # Generated reply
        print(f"\n✉️  GENERATED REPLY:")
        print(f"   To: {original_email['sender_name']} <{original_email['sender']}>")
        print(f"   From: {reply['from']}")
        print(f"   Subject: {reply['subject']}")
        print(f"\n   Message:\n")
        for line in reply['reply_body'].split('\n'):
            print(f"   {line}")
        
        print(f"\n   Generated at: {reply['generated_at']}")
        print("-" * 100)

print("\n" + "=" * 100)


📨 DETAILED VIEW OF GENERATED REPLIES


                                              REPLY 1                                               

📬 ORIGINAL EMAIL:
   From: Rajesh Kumar <rajesh.kumar@techsolutions.in>
   Subject: Project Update - Q4 Goals and Team Alignment
   Type: BUSINESS | Priority: HIGH

   Message:
   Hi team, I hope this email finds you well. As we approach the end of Q3, I wanted to reach out to discuss our Q4 goals and how we can better align our efforts across all departments. We've made significant progress on the cloud migration project, but there are still some challenges with the timeline that need to be addressed. I'd like to schedule a meeting next week to review our current status, identify any bottlenecks, and ensure everyone is on the same page regarding priorities. Please let me know your availability for Tuesday or Wednesday afternoon. Looking forward to a productive discussion.

✉️  GENERATED REPLY:
   To: Rajesh Kumar <rajesh.kumar@techsolutions.in>


---

## Part 5: Save Replies to CSV

### 📚 What You'll Learn:
- Data persistence for generated replies
- Using pandas for CSV export
- Creating audit trails for email automation

1. Audit Trial
2. Quality Reviews
3. Analytics
4. Integration

In [17]:
def save_replies_to_csv(replies, filename="email_replies_log.csv"):
    """
    Save all generated replies to a CSV file
    
    Args:
        replies (list): List of reply dictionaries
        filename (str): Output CSV filename
    
    Returns:
        DataFrame: Saved replies as a DataFrame
    """
    if not replies:
        print("❌ No replies to save")
        return None
    
    try:
        # Filter successful replies only
        successful_replies = [r for r in replies if r.get("status") == "generated"]
        
        if successful_replies:
            df_replies = pd.DataFrame(successful_replies)
            df_replies.to_csv(filename, index=False)
            print(f"✅ {len(successful_replies)} replies saved to '{filename}'")
            return df_replies
        else:
            print("❌ No successful replies to save")
            return None
    
    except Exception as e:
        print(f"❌ Error saving replies: {str(e)}")
        return None

# Save the generated replies
replies_df = save_replies_to_csv(generated_replies)

if replies_df is not None:
    print("\n📊 CSV Preview:")
    print("=" * 100)
    print(replies_df[['original_email_id', 'to', 'subject', 'status', 'generated_at']].to_string())
    print("=" * 100)

✅ 5 replies saved to 'email_replies_log.csv'

📊 CSV Preview:
   original_email_id                              to                                                                subject     status         generated_at
0                  1   rajesh.kumar@techsolutions.in                       Re: Project Update - Q4 Goals and Team Alignment  generated  2025-12-28 11:50:07
1                  2     priya.sharma@clientcorp.com    Re: Feedback on Proposal - Pricing and Implementation Clarification  generated  2025-12-28 11:50:13
2                  3              hr@innovatetech.in               Re: Annual Performance Review Schedule - Action Required  generated  2025-12-28 11:50:18
3                  4  arjun.verma@globalpartners.com       Re: Strategic Partnership Opportunity - AI Solutions Integration  generated  2025-12-28 11:50:21
4                  5      events@indiatechsummit.com  Re: Early Bird Registration Extended - Save 30% on Tech Summit Passes  generated  2025-12-28 11:50:27


---

## Part 6: Advanced Feature - Tone Personalization

### 📚 What You'll Learn:
- How to customize LLM output with tone parameters
- Creating dynamic prompts with style instructions
- Comparing different communication styles

Sender Name: {sender_name}
Email Type: {email_type}
Priority Level: {priority}
Original Email Subject: {subject}
Original Email Body: {body}
Your Name: {your_name}
Your Title: {your_title}
Company: {company}

In [18]:
# TODO: Define personalization styles and create advanced prompt
# HINT: Dictionary of tone descriptions + new prompt template with {style_description}

# Step 1: Define 5 personalization styles
personalization_styles = {
    "formal": "Use very formal, professional language with proper salutations",
    "friendly": "Use warm, friendly tone while maintaining professionalism",
    "brief": "Keep the response concise and to the point",
    "detailed": "Provide comprehensive and detailed response with examples",
    "empathetic": "Show understanding and empathy for the sender's situation"
}


# Step 2: Create advanced prompt template with tone
advanced_email_template = """You are a professional email assistant. Generate a personalized, 
professional email reply based on the following context.

Sender Name: {sender_name}
Email Type: {email_type}
Priority Level: {priority}
Original Email Subject: {subject}
Original Email Body: {body}
Your Name: {your_name}
Your Title: {your_title}
Company: {company}

Respnse style: {style_description}

Generate a professional, personalized reply that:
1. Addresses the sender by name
2. Acknowledge their emails properly
3. Provide a thoughtful response that is relevent to the email type
4. Maintain a professional tone
5. Include the clear call-to-action or next steps

Reply Email:"""

# Step 3: Create function for personalized replies
def generate_personalized_reply(email_data, your_name="Sarvesh", 
                               your_title="Project Manager", company="Learn With Sarvesh", 
                               tone="formal"):
    
    """
    Generate a personalized email reply with custom tone
    
    Args:
        email_data (dict): Email information
        your_name (str): Your name
        your_title (str): Your title
        company (str): Your company
        tone (str): Desired tone (formal/friendly/brief/detailed/empathetic)
    
    Returns:
        dict: Generated reply with tone information
    """
    # Get style description
    style_description = personalization_styles.get(tone, personalization_styles["formal"])
    
    # Create prompt with tone
    advanced_prompt = PromptTemplate(
        input_variables=["sender_name", "email_type", "priority", "subject", "body", 
                        "your_name", "your_title", "company", "style_description", "tone"],
        template=advanced_email_template
    )

    # Create chain
    advanced_chain = advanced_prompt | llm
    
    try:
        result = advanced_chain.invoke({
            "sender_name": email_data.get("sender_name", ""),
            "email_type": email_data.get("email_type", ""),
            "priority": email_data.get("priority", ""),
            "subject": email_data.get("subject", ""),
            "body": email_data.get("body", ""),
            "your_name": your_name,
            "your_title": your_title,
            "company": company,
            "style_description": style_description,
            "tone": tone
        })
        
        reply = result.content if hasattr(result, 'content') else str(result)
        return {"tone": tone, "reply": reply.strip(), "status": "success"}
    
    except Exception as e:
        return {"tone": tone, "error": str(e), "status": "failed"}

print("✅ Tone personalization function created")
print("\n📝 Available tones:")
for tone, description in personalization_styles.items():
    print(f"   • {tone.upper()}: {description}")

✅ Tone personalization function created

📝 Available tones:
   • FORMAL: Use very formal, professional language with proper salutations
   • FRIENDLY: Use warm, friendly tone while maintaining professionalism
   • BRIEF: Keep the response concise and to the point
   • DETAILED: Provide comprehensive and detailed response with examples
   • EMPATHETIC: Show understanding and empathy for the sender's situation


### 🎯 Demo: Compare Different Tones

Let's generate replies to the same email using different tones to see the impact.

In [19]:
# Demo: Generate replies with different tones for the same email
demo_email = sample_emails[1]  # Client feedback email

print("🎯 TONE COMPARISON DEMO")
print("=" * 100)
print(f"\n📧 Original Email from {demo_email['sender_name']}:")
print(f"   Subject: {demo_email['subject']}")
print(f"   Message: {demo_email['body']}")
print("\n" + "=" * 100)

demo_tones = ["formal", "friendly", "brief"]

for tone in demo_tones:
    print(f"\n✉️  TONE: {tone.upper()}")
    print("-" * 100)
    
    result = generate_personalized_reply(
        demo_email,
        tone=tone,
        your_name="Sarah Johnson",
        your_title="Project Manager",
        company="TechCorp"
    )
    
    if result["status"] == "success":
        print(f"\n{result['reply']}\n")
    else:
        print(f"Error: {result.get('error')}")
    
    print("-" * 100)

print("\n💡 Notice how the same email gets different responses based on tone!")

🎯 TONE COMPARISON DEMO

📧 Original Email from Priya Sharma:
   Subject: Feedback on Proposal - Pricing and Implementation Clarification
   Message: Dear team, Thank you so much for sending over the detailed proposal for the enterprise software solution. We've reviewed it thoroughly with our stakeholders, and overall, we're very impressed with the features and timeline you've outlined. However, we do have some questions regarding the pricing model, particularly around the tiered structure and what's included in each tier. Could you please clarify the differences between the Standard and Premium packages? Additionally, we'd like to understand the implementation timeline better - specifically, how the phases would work for our organization size of 500+ employees. We're also interested in knowing if there's any flexibility in customizing certain modules to fit our specific workflow requirements. Would it be possible to schedule a call this week to discuss these points in detail?


✉️  TONE

### Threading:
Thread != CPU Thread

One email

---

## Part 7: Conversation Memory - Thread-Aware Replies

### 📚 What You'll Learn:
- Implementing conversation memory in LangChain
- Maintaining context across email threads
- Understanding different memory types

### Diff Types of Memory:
1. Conversation Buffer Memory -> ChatBot conversations, during the chat
2. Conversation Summary Memory -> 100 messages in chatgpt | I will summarize the past messages in less than 100 characters
3. Conversation Window Memory -> Sliding Window (100 Messages) ; if the there is 101st message, the 1st message is deleted

1. RunnableWithMessageHistory
2. InMemoryChatMessageHistory
3. Alternative storeges like Redis for caching, Postgressql, or other backend storages


RunnableWithMessageHistory with Sessions Ids to maintain sperate contexts per thread.

No Memory -> No past histories
Memory store -> Thread-Based Memory Store (Remember previous emails in the same conversation)

### One Email Thread = One Memory Timeline

thread_message_histories -> Conversation history

- First message in a thread -> Create Memory
- Subsequent Messages -> Reuse te same memory

## Memory Injection Point 

In [20]:
# Initialize thread-based message history storage

thread_message_histories = {}  # Dictionary to store message history for each thread 

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """
    Retrieve or create a message history for a given session/thread.
    
    Args:
        session_id (str): Unique identifier for the conversation thread
    
    Returns:
        BaseChatMessageHistory: The message history for this session
    """
    # First message in a thread → create memory
    # Next messages → reuse same memory
    
    if session_id not in thread_message_histories:
        thread_message_histories[session_id] = InMemoryChatMessageHistory()
    return thread_message_histories[session_id]

# Create prompt template with memory placeholder
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional email assistant. Generate clear, professional responses that reference relevant prior context when appropriate."),
    MessagesPlaceholder(variable_name="history"),  # This is where conversation history goes, it injects all previous messages from this thread here.
    ("human",
     "Sender: {sender_name}\n"
     "Email Type: {email_type}\n"
     "Priority: {priority}\n"
     "Subject: {subject}\n"
     "Body: {body}\n"
     "Tone: {tone}\n"
     "Your Name: {your_name}\n"
     "Your Title: {your_title}\n"
     "Company: {company}\n\n"
     "Draft a professional reply.")
]) # Structured email metadata, clear instructions, formatting

# Create base chain
base_chain = memory_prompt | llm

# Wrap chain with message history - For this session_id, where should I store messages?
chain_with_history = RunnableWithMessageHistory(
    base_chain,  # The LLM pipeline to wrap.
    get_session_history,   # Your memory lookup function.
    input_messages_key="body", # This field is the new human message.
    history_messages_key="history" # Inject stored messages into the history placeholder. 
)

def generate_reply_with_memory(   # The Orchestrator
    email_data,
    thread_id = None,
    your_name = "Sarvesh",
    your_title = "Delivery Manager",
    company = "Learn With Sarvesh",
    tone = "formal"
):
    """
    Generate an email reply with conversation memory for thread context using RunnableWithMessageHistory.
    
    Args:
        email_data (dict): Email information
        thread_id (str): Unique thread identifier (defaults to subject)
        your_name (str): Your name
        your_title (str): Your title
        company (str): Your company
        tone (str): Response tone
    
    Returns:
        dict: Generated reply with thread context
    """
    # Determine thread key (use thread_id or fall back to subject)
    thread_key = thread_id or email_data.get("subject") or f"thread-{email_data.get('id', '0')}"
    
    # Get current message count for this thread (before adding new message)
    current_history = get_session_history(thread_key)
    previous_message_count = len(current_history.messages)
    
    # Generate reply with context using RunnableWithMessageHistory
    result = chain_with_history.invoke(
        {
            "sender_name": email_data.get("sender_name", ""),
            "email_type": email_data.get("email_type", ""),
            "priority": email_data.get("priority", ""),
            "subject": email_data.get("subject", ""),
            "body": email_data.get("body", ""),
            "tone": tone,
            "your_name": your_name,
            "your_title": your_title,
            "company": company,
        },
        config={"configurable": {"session_id": thread_key}}
    )
    
    reply = result.content if hasattr(result, "content") else str(result)
    
    return {
        "original_email_id": email_data.get("id"),
        "from": your_name,
        "to": email_data.get("sender"),
        "subject": f"Re: {email_data.get('subject')}",
        "reply_body": reply.strip(),
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "status": "generated",
        "thread_key": thread_key,
        "memory_turns": previous_message_count // 2  # Number of exchanges before this one
    }

print("✅ Memory-enabled email reply function created")
print("\n📝 Memory Configuration:")
print("   • Type: RunnableWithMessageHistory (modern LangChain pattern)")
print("   • Storage: InMemoryChatMessageHistory per thread")
print("   • Scope: Per-thread (separate memory for each email thread)")

✅ Memory-enabled email reply function created

📝 Memory Configuration:
   • Type: RunnableWithMessageHistory (modern LangChain pattern)
   • Storage: InMemoryChatMessageHistory per thread
   • Scope: Per-thread (separate memory for each email thread)


In [ ]:
thread_key = thread_id or email_data.get("subject") or f"thread-{email_data.get('id', '0')}"

thread_id = abc123 # Thread key = ticketno: 0001

Thread Key as the Email Subject: "Invoice no 123 follow-ip"

In [ ]:
# thread-{email_data.get('id', '0')}" -> Called as thread-123

"id" : 123

## Whenever an email we received normalize the subject
- Normal if Re: Remove the first 2 chracterd and check; FW: -> Remmove and check it 

### 🎯 Demo: Multi-Turn Email Thread

Watch how the assistant maintains context across 3 email exchanges in the same thread.

### 💡 What is Thread Memory?

**Thread = Email Conversation**
- Each unique email conversation gets its own memory
- Example: Customer asking about pricing (Thread 1) vs. Support ticket about bugs (Thread 2)
- Thread 1's memory: "They asked about pricing", "They need 500 users"
- Thread 2's memory: "Error code 500", "MySQL timeout"
- These memories are COMPLETELY SEPARATE ✓

**Why Thread Isolation Matters:**
- When replying to a support ticket, the AI won't accidentally mention pricing from another thread
- Each conversation maintains its own context
- Prevents confusion and ensures relevant replies


In [ ]:
# Demo: Same thread across three emails to show memory carryover (Indian Business Context)
demo_thread = [
    {
        "id": 201,
        "sender": "vikram.sharma@fintech-mumbai.com",
        "sender_name": "Vikram Sharma",
        "subject": "Enterprise Software Proposal - Pricing and Support Query",
        "body": "Hello, we are very interested in your solution for our Mumbai operations. Can you clarify the different pricing tiers available and what support is included in each package? We also need to know if GST is additional to the quoted price.",
        "email_type": "client",
        "priority": "high",
    },
    {
        "id": 202,
        "sender": "vikram.sharma@fintech-mumbai.com",
        "sender_name": "Vikram Sharma",
        "subject": "Enterprise Software Proposal - Pricing and Support Query",
        "body": "Thank you for the detailed information you provided. That was very helpful. I have a couple of follow-up questions: What are your payment terms? Do you offer any discount if we commit to an annual billing cycle? We usually finalize our budget between October and December.",
        "email_type": "client",
        "priority": "medium",
    },
    {
        "id": 203,
        "sender": "vikram.sharma@fintech-mumbai.com",
        "sender_name": "Vikram Sharma",
        "subject": "Enterprise Software Proposal - Pricing and Support Query",
        "body": "Perfect, those terms work well for us. Now, can we schedule a demo call sometime next week? We are in IST timezone, so ideally between 10 AM and 12 PM would work best for our team.",
        "email_type": "client",
        "priority": "medium",
    },
]

print("🚀 MEMORY DEMO: Same Thread, Multiple Exchanges")
print("=" * 100)
print("\n💡 Watch how each reply references previous exchanges in the thread!\n")

for i, email in enumerate(demo_thread, 1):
    print(f"\n{'EMAIL ' + str(i) + ' IN THREAD':^100}")
    print("-" * 100)
    print(f"From: {email['sender_name']}")
    print(f"Subject: {email['subject']}")
    print(f"Message: {email['body']}\n")
    
    # Generate reply with memory
    reply = generate_reply_with_memory(
        email,
        thread_id="proposal-followup",  # All emails share same thread
        tone="friendly"
    )
    
    print(f"{'REPLY ' + str(i):^100}")
    print("-" * 100)
    print(f"📊 Context: {reply['memory_turns']} previous exchange(s) in memory")
    print(f"\n{reply['reply_body']}\n")
    print("=" * 100)

# Show memory statistics
thread_history = get_session_history('proposal-followup')
total_messages = len(thread_history.messages)

print("\n✅ Demo Complete!")
print(f"\n📊 Thread Statistics:")
print(f"   • Thread ID: proposal-followup")
print(f"   • Total messages in memory: {total_messages}")
print(f"   • Exchanges (back-and-forth): {total_messages // 2}")
print(f"\n💡 Notice how Reply 2 and 3 reference earlier parts of the conversation!")

🚀 MEMORY DEMO: Same Thread, Multiple Exchanges

💡 Watch how each reply references previous exchanges in the thread!


                                         EMAIL 1 IN THREAD                                          
----------------------------------------------------------------------------------------------------
From: Vikram Sharma
Subject: Enterprise Software Proposal - Pricing and Support Query
Message: Hello, we are very interested in your solution for our Mumbai operations. Can you clarify the different pricing tiers available and what support is included in each package? We also need to know if GST is additional to the quoted price.

                                              REPLY 1                                               
----------------------------------------------------------------------------------------------------
📊 Context: 0 previous exchange(s) in memory

Subject: Re: Enterprise Software Proposal - Pricing and Support Query

Dear Vikram,

Thank you for y

In [ ]:
# Demo: Multiple Parallel Threads - Showing How Different Conversations Stay Isolated
print("🎯 ADVANCED DEMO: Different Threads Are Kept Separate")
print("=" * 100)
print("\n💡 Watch how memory is maintained independently for each thread!\n")

# Thread 1: Software Proposal Discussion
thread_1_emails = [
    {
        "id": 301,
        "sender": "vikram.sharma@fintech-mumbai.com",
        "sender_name": "Vikram Sharma",
        "subject": "Software Proposal Question",
        "body": "Hello, we have questions about your pricing model and implementation timeline.",
        "email_type": "client",
        "priority": "high"
    },
    {
        "id": 302,
        "sender": "vikram.sharma@fintech-mumbai.com",
        "sender_name": "Vikram Sharma",
        "subject": "Software Proposal Question",
        "body": "Thanks for the proposal details. Can we schedule a demo?",
        "email_type": "client",
        "priority": "high"
    }
]

# Thread 2: Support Ticket Discussion (different customer, different context)
thread_2_emails = [
    {
        "id": 401,
        "sender": "support@acmetech.com",
        "sender_name": "Support Team",
        "subject": "Technical Issue - API Integration",
        "body": "We're having trouble integrating your API with our system. Error code: 500. Can you help?",
        "email_type": "support",
        "priority": "high"
    },
    {
        "id": 402,
        "sender": "support@acmetech.com",
        "sender_name": "Support Team",
        "subject": "Technical Issue - API Integration",
        "body": "We tried the solution you suggested, but still getting the same error. Any other ideas?",
        "email_type": "support",
        "priority": "high"
    }
]

# Thread 3: Partnership Discussion (yet another conversation)
thread_3_emails = [
    {
        "id": 501,
        "sender": "bd@globalpartners.com",
        "sender_name": "BD Manager",
        "subject": "Partnership Opportunity",
        "body": "I'm interested in exploring a partnership between our companies. We have 200+ potential clients.",
        "email_type": "business",
        "priority": "medium"
    },
    {
        "id": 502,
        "sender": "bd@globalpartners.com",
        "sender_name": "BD Manager",
        "subject": "Partnership Opportunity",
        "body": "Following up on our partnership discussion. When would be a good time to connect?",
        "email_type": "business",
        "priority": "medium"
    }
]

# Process all three threads
all_threads = [
    ("software-proposal", thread_1_emails, "Proposal Discussion"),
    ("api-support-ticket", thread_2_emails, "Technical Support"),
    ("partnership-deal", thread_3_emails, "Business Partnership")
]

thread_results = {}

for thread_id, emails, thread_name in all_threads:
    print(f"\n{'THREAD: ' + thread_name.upper():^100}")
    print("=" * 100)
    
    thread_results[thread_id] = []
    
    for i, email in enumerate(emails, 1):
        print(f"\n📧 EMAIL {i} in thread '{thread_id}'")
        print("-" * 100)
        print(f"From: {email['sender_name']}")
        print(f"Message: {email['body']}\n")
        
        # Generate reply with memory - SAME thread_id keeps memory isolated
        reply = generate_reply_with_memory(
            email,
            thread_id=thread_id,  # KEY: Each thread has its own memory
            tone="professional"
        )
        
        thread_results[thread_id].append(reply)
        
        print(f"{'REPLY ' + str(i):^50}")
        print("-" * 100)
        print(f"📊 Memory context: {reply['memory_turns']} previous exchange(s) in this specific thread")
        print(f"\n{reply['reply_body'][:300]}...\n")

# Show all thread memories separately
print("\n" + "=" * 100)
print("🧠 MEMORY ISOLATION ANALYSIS")
print("=" * 100)

for thread_id, _, thread_name in all_threads:
    history = get_session_history(thread_id)
    print(f"\n📌 Thread: {thread_name} (ID: '{thread_id}')")
    print(f"   • Messages in memory: {len(history.messages)}")
    print(f"   • Back-and-forth exchanges: {len(history.messages) // 2}")
    print(f"   • Memory status: ✅ ISOLATED (kept separate from other threads)")

print("\n" + "=" * 100)
print("💡 KEY INSIGHT: Each thread maintains its own independent memory timeline!")
print("   ✅ Thread 'software-proposal' knows about pricing and demos")
print("   ✅ Thread 'api-support-ticket' knows about API errors")
print("   ✅ Thread 'partnership-deal' knows about business partnership")
print("   ✅ Zero cross-contamination between threads!")
print("\n🎯 Real-world example:")
print("   When replying to a support ticket (api-support-ticket),")
print("   the AI doesn't accidentally mention software pricing")
print("   because those are in separate threads with separate memories!")
print("=" * 100)

1. Conversation Chain - Response Chain

Memory  -> Response Chain (prompt | LLM) -> Memory

### 🧠 Understanding Memory Types

| Storage Type | Use Case | Pros | Cons |
|-------------|----------|------|------|
| **InMemoryChatMessageHistory** | Development, short sessions | Fast, simple, no setup | Lost on restart, not scalable |
| **RedisChatMessageHistory** | Production, distributed systems | Persistent, scalable | Requires Redis server |
| **PostgresChatMessageHistory** | Enterprise, data persistence | Durable, queryable | Requires database setup |

### ✅ When to Use Memory

**Use memory when:**
- Handling email threads with multiple back-and-forth exchanges
- Customer support scenarios where context matters
- Follow-up emails that reference previous discussions

**Skip memory when:**
- Processing standalone, unrelated emails
- Batch processing where emails don't form threads
- Memory/cost constraints are critical

# 🎓 Complete Summary & Key Takeaways

## ✅ What You Built Today:

### 1. **Automated Email Reply System**
   - ✅ Integrated LangChain with OpenAI for intelligent email generation
   - ✅ Designed context-rich prompts with personalization variables
   - ✅ Built reusable chains using LCEL (LangChain Expression Language)
   - ✅ Key Function: `generate_email_reply(email_data)`

### 2. **Batch Processing Pipeline**
   - ✅ Processed multiple emails efficiently with error handling
   - ✅ Generated professional, context-aware replies
   - ✅ Saved results to CSV for audit trails and analysis
   - ✅ Key Function: Looping through `sample_emails`

### 3. **Tone Personalization**
   - ✅ Created 5 different communication styles:
     - **Formal**: Professional and structured
     - **Friendly**: Warm and approachable
     - **Brief**: Concise and to the point
     - **Detailed**: Comprehensive with examples
     - **Empathetic**: Shows understanding and care
   - ✅ Key Function: `generate_personalized_reply(email_data, tone="...")`

### 4. **Conversation Memory (Thread-Aware Replies)**
   - ✅ Implemented thread-aware memory to maintain context
   - ✅ Used `RunnableWithMessageHistory` for context preservation
   - ✅ Demonstrated multi-turn conversations with context carryover
   - ✅ Showed how different threads stay completely isolated
   - ✅ Key Function: `generate_reply_with_memory(email_data, thread_id="...")`

### 5. **Real Email Processing (.eml Files)**
   - ✅ Parse emails exported from Gmail/Outlook
   - ✅ Convert .eml binary format to JSON
   - ✅ Batch process large email datasets
   - ✅ Key Functions: `parse_eml_file()`, `parse_eml_directory()`

---

## 🎯 Architecture Overview

```
┌─────────────────────────────────────┐
│         Email Source                │
│      (Gmail/.eml files)             │
└────────────────┬────────────────────┘
                 │
                 ▼
┌─────────────────────────────────────┐
│        Email Parser                 │
│     (parse_eml_file)                │
└────────────────┬────────────────────┘
                 │
                 ▼
┌─────────────────────────────────────┐
│        Email Data                   │
│       (JSON Format)                 │
└────────────────┬────────────────────┘
                 │
                 ▼
┌─────────────────────────────────────┐
│    LangChain Processing             │
│  ┌─────────────────────────────┐    │
│  │  PromptTemplate             │    │
│  │  + ChatOpenAI LLM           │    │
│  │  + Tone Customization       │    │
│  │  + Thread Memory            │    │
│  └──────────┬──────────────────┘    │
└─────────────┼──────────────────────┘
              │
              ▼
┌─────────────────────────────────────┐
│      Generated Reply                │
│   (Professional Text)               │
└─────────────────────────────────────┘
```

---

## 📚 Learning Path

**Beginner Level:**
- Use `generate_email_reply()` for basic replies
- Understand the prompt template structure
- Process single emails

**Intermediate Level:**
- Use `generate_personalized_reply()` with different tones
- Process multiple emails in a batch
- Save results to CSV for analysis

**Advanced Level:**
- Use `generate_reply_with_memory()` for multi-turn conversations
- Manage separate thread memories
- Parse real .eml files from Gmail/Outlook
- Integrate into production applications

---

## 🔑 Key Concepts

| Concept | Definition | Example |
|---------|-----------|---------|
| **Thread** | A conversation between sender and recipient | "Support Ticket #123" or "Project Proposal ABC" |
| **Memory** | AI's ability to recall previous messages in a thread | AI remembers customer asked about pricing |
| **Thread Isolation** | Each thread has separate memory | Support thread won't interfere with sales thread |
| **LCEL** | LangChain Expression Language for chaining | `prompt \| llm` pipes prompt into LLM |
| **Tone** | Communication style/personality | formal, friendly, brief, detailed, empathetic |
| **.eml Format** | Standard email file format | Works with Gmail, Outlook, Apple Mail |

---

## 🚀 Real-World Applications

1. **Customer Support Automation**
   - Automatically reply to common support tickets
   - Maintain conversation context
   - Escalate complex issues to humans

2. **Sales Email Responses**
   - Generate personalized responses to inquiries
   - Adjust tone based on client type
   - Remember previous interactions

3. **Internal Communications**
   - Auto-respond to team emails
   - Route emails to appropriate departments
   - Maintain professional communication standards

4. **HR and Administration**
   - Respond to leave requests
   - Answer policy questions
   - Schedule interviews

---

## 🎉 Congratulations!

You've successfully built a **production-ready AI Email Assistant** with:
- ✅ LLM-powered reply generation
- ✅ Tone personalization (5 different styles)
- ✅ Conversation memory (thread-aware)
- ✅ Batch processing capabilities
- ✅ Data persistence (CSV export)
- ✅ Real email parsing (.eml support)

**This is a complete professional solution that could be deployed to production!**

---

## 📖 Next Steps

**Immediate:**
1. Run through all cells and understand the output
2. Modify the sample emails to test different scenarios
3. Try different tones and see how the responses change
4. Export your own emails and parse them

**Short-term:**
1. Build a simple Flask/FastAPI endpoint
2. Add database support (PostgreSQL)
3. Implement email sending (smtplib)
4. Create a web dashboard for monitoring

**Long-term:**
1. Deploy to cloud (AWS, Google Cloud, Azure)
2. Add fine-tuning with your own email examples
3. Integrate with email calendars
4. Build team collaboration features

---

## 📞 Support & Resources

- **LangChain Documentation**: https://python.langchain.com/
- **OpenAI API Docs**: https://platform.openai.com/docs/
- **Email Parsing**: https://docs.python.org/3/library/email.html
- **Your Instructor**: Sarvesh (Learn With Sarvesh)

---

## 🎁 Challenge Exercises

### Challenge 1: Multi-tone Responses
Generate replies in all 5 tones for the same email and compare them.

### Challenge 2: Thread Analysis
Create 5 different customer threads and show how memory is maintained.

### Challenge 3: .eml Batch Processing
Export 10 emails from Gmail and process them with your parser.

### Challenge 4: Custom Tone
Create your own tone (e.g., "humorous", "casual") and add it to the personalization_styles dict.

### Challenge 5: Integration
Build a simple Flask API that accepts email JSON and returns AI-generated replies.

---

**Happy Learning! 🚀**

---

## 🎁 BONUS: Extract Emails from .eml Files (Production Path)

### 📚 What You'll Learn:
- Understanding .eml file format (standard email format)
- Parsing .eml files with Python's `email` library
- Extracting key fields and converting to JSON
- How to integrate real emails from Gmail/Outlook

### 🎤 SPEAKER NOTE:
**What are .eml files?**
- Standard email format (works with Gmail, Outlook, Apple Mail, etc.)
- Plain text files containing email metadata and content
- Easy to export and parse programmatically
- Perfect for building production email systems

**Real-world workflow:**
1. User exports emails from Gmail/Outlook as .eml files
2. Your script parses them automatically
3. Converts to JSON format we've been using
4. Feeds into your AI Email Assistant
5. Generates and logs replies

**This is exactly what enterprise email systems do!**


In [ ]:
# 📧 BONUS SECTION: Parse Real Email Files (.eml Format)
# ==========================================================
# This section shows how to work with real emails exported from Gmail/Outlook
# .eml files are standard email format used by all major email providers

# Step 1: Import required libraries for email parsing
import email
from email.parser import BytesParser
from email.policy import default
import mimetypes
from pathlib import Path

print("✅ Email parsing libraries imported")
print("\n📚 Available email parsing tools:")
print("   • email.parser - Parse .eml files (standard Python library)")
print("   • email.policy - Handle email standards (RFC 5322 compliant)")
print("   • pathlib - Work with file paths in a cross-platform way")
print("\n🎯 Use Cases:")
print("   • Process exported Gmail emails")
print("   • Parse emails from Outlook/Apple Mail")
print("   • Batch import large email datasets")
print("   • Build production email workflows")

### How .eml Files Work

**Step 1: Export from Gmail**
1. Open Gmail
2. Select email → More (⋮) → Download Message
3. File saved as `.eml` (Example: `email_name.eml`)

**Step 2: .eml File Structure**
```
From: sender@example.com
To: recipient@example.com
Subject: Email Subject Here
Date: Mon, 26 Dec 2024 10:30:00 +0000
Content-Type: text/plain; charset="UTF-8"

This is the email body content.
It can be multiple lines.
```

**Step 3: Parse with Python**
Your script reads this file → Extracts fields → Converts to JSON


In [ ]:
def parse_eml_file(eml_file_path, email_id=None, email_type="business", priority="medium"):
    """
    Parse a single .eml file and convert to our JSON email format.
    
    This function handles the technical details of reading and extracting
    information from email files in standard .eml format.
    
    The Process:
    1. Open and read the binary .eml file
    2. Parse it using BytesParser (handles all email standards)
    3. Extract key fields: sender, subject, body
    4. Convert to our standard JSON format
    5. Return structured data ready for AI processing
    
    Args:
        eml_file_path (str): Path to the .eml file
                            Example: "emails/proposal.eml"
        email_id (int): Unique email ID (auto-generated if not provided)
        email_type (str): Type of email - one of:
            - "business": Internal or general business communication
            - "client": Customer or external client emails
            - "hr": Human resources related emails
            - "support": Technical or customer support tickets
            - "notification": System notifications or announcements
        priority (str): Priority level - one of:
            - "high": Requires urgent attention
            - "medium": Normal priority
            - "low": Can be handled later
    
    Returns:
        dict: Email in our standard JSON format with keys:
            - id: Unique identifier
            - sender: Email address of sender
            - sender_name: Full name of sender
            - subject: Email subject line
            - body: Email message content (plain text)
            - email_type: Category of email
            - priority: Importance level
            
    Example:
        >>> email = parse_eml_file("proposal.eml", email_type="client", priority="high")
        >>> print(email['sender_name'])
        'John Smith'
    """
    try:
        # Read and parse the .eml file using standard Python library
        with open(eml_file_path, 'rb') as f:
            msg = BytesParser(policy=default).parse(f)
        
        # Extract sender information
        sender_email = msg.get('From', 'unknown@example.com')
        # Split name from email: "John Smith <john@example.com>" → "John Smith"
        sender_name = msg.get('From', 'Unknown Sender').split('<')[0].strip()
        
        # Extract subject line (or use default if missing)
        subject = msg.get('Subject', '(No Subject)')
        
        # Extract email body (plain text)
        # Emails can be multipart (text + HTML) or simple text
        body = ""
        if msg.is_multipart():
            # Multipart email: loop through parts and find plain text
            for part in msg.iter_parts():
                if part.get_content_type() == "text/plain":
                    body = part.get_content()
                    break
        else:
            # Simple email: just get the content
            body = msg.get_content()
        
        # Clean up body: remove extra whitespace
        body = body.strip() if body else "(No content)"
        
        # Return structured email data
        return {
            "id": email_id or hash(sender_email + subject) % 10000,
            "sender": sender_email,
            "sender_name": sender_name,
            "subject": subject,
            "body": body,
            "email_type": email_type,
            "priority": priority
        }
    
    except Exception as e:
        # Handle errors gracefully
        print(f"❌ Error parsing {eml_file_path}: {str(e)}")
        return None

print("✅ EML parser function created successfully")
print("\n📝 Function signature:")
print("   parse_eml_file(eml_file_path, email_id, email_type, priority)")
print("\n💡 This function extracts:")
print("   • Sender email and full name")
print("   • Subject line")
print("   • Email body (plain text content)")
print("   • Converts to our standard JSON format")
print("\n🎯 Perfect for batch processing Gmail exports!")

In [ ]:
def parse_eml_directory(directory_path):
    """
    Batch process: Parse ALL .eml files in a directory.
    
    This is the production-ready function for processing large email exports
    from Gmail, Outlook, or any email client that supports .eml export.
    
    How it works:
    1. Scans the directory for all .eml files
    2. Parses each file using parse_eml_file()
    3. Collects results into a list
    4. Returns ready-to-process email dataset
    
    Args:
        directory_path (str): Path to folder containing .eml files
                            Example: "./downloaded_emails/"
    
    Returns:
        list: List of emails in JSON format, ready for AI processing
              Each item has: id, sender, sender_name, subject, body, email_type, priority
    
    Example:
        >>> emails = parse_eml_directory("./my_emails/")
        >>> len(emails)
        5
        >>> emails[0]['sender_name']
        'John Smith'
    """
    emails = []
    # Find all .eml files in the directory (non-recursive)
    eml_files = Path(directory_path).glob('*.eml')
    
    # Parse each file
    for idx, eml_file in enumerate(eml_files, 1):
        email_data = parse_eml_file(eml_file, email_id=idx)
        if email_data:
            emails.append(email_data)
    
    return emails

print("✅ Batch EML parser function created")
print("\n📝 Function: parse_eml_directory(directory_path)")
print("\n🔧 Capabilities:")
print("   • Finds all .eml files in a folder")
print("   • Parses each one automatically")
print("   • Returns list of JSON emails")
print("   • Ready to feed into AI Email Assistant")
print("   • Scales to thousands of emails")
print("\n🚀 Real-World Workflow:")
print("   1. Export folder from Gmail: ⬇️")
print("   2. Parse with this function: ⚙️")
print("   3. Feed to AI Assistant: 🤖")
print("   4. Get automated replies: 📧")

### 🎯 Demo: How to Use in Real Workflow

**Scenario:** Student exports 5 emails from Gmail as .eml files

**Your code does this:**
```python
# Instead of hardcoded emails...
sample_emails = [...]  # This is what we did today

# You'd do this in production...
eml_folder = "downloaded_emails/"
emails_from_gmail = parse_eml_directory(eml_folder)

# Then feed to AI Assistant
for email in emails_from_gmail:
    reply = generate_email_reply(email)
```

**That's it! Real Gmail emails → Your AI Assistant → Professional replies!**


In [ ]:
# 📧 CREATE DEMO .eml FILES FOR TESTING\n# ========================================\n# In real scenarios, you would export these from Gmail/Outlook\n# Here we're creating sample files to demonstrate the process\n\n# Sample .eml file 1: Team project update (realistic format)\nsample_eml_content_1 = \"\"\"From: rajesh.kumar@techsolutions.in\nTo: your_email@company.com\nSubject: Project Update - Q4 Goals and Team Alignment\nDate: Mon, 26 Dec 2024 10:30:00 +0000\nContent-Type: text/plain; charset=\"UTF-8\"\n\nHi team, I hope this email finds you well. As we approach the end of Q3, I wanted to reach out to discuss our Q4 goals and how we can better align our efforts across all departments. We've made significant progress on the cloud migration project, but there are still some challenges with the timeline that need to be addressed. I'd like to schedule a meeting next week to review our current status, identify any bottlenecks, and ensure everyone is on the same page regarding priorities. Please let me know your availability for Tuesday or Wednesday afternoon. Looking forward to a productive discussion.\"\"\"\n\n# Sample .eml file 2: Client proposal feedback\nsample_eml_content_2 = \"\"\"From: priya.sharma@clientcorp.com\nTo: your_email@company.com\nSubject: Feedback on Proposal - Pricing and Implementation Clarification\nDate: Mon, 26 Dec 2024 11:45:00 +0000\nContent-Type: text/plain; charset=\"UTF-8\"\n\nDear team, Thank you so much for sending over the detailed proposal for the enterprise software solution. We've reviewed it thoroughly with our stakeholders, and overall, we're very impressed with the features and timeline you've outlined. However, we do have some questions regarding the pricing model, particularly around the tiered structure and what's included in each tier. Could you please clarify the differences between the Standard and Premium packages? Additionally, we'd like to understand the implementation timeline better. Would it be possible to schedule a call this week to discuss these points in detail?\"\"\"\n\n# Create directory and files\nimport tempfile\nimport shutil\n\n# Create temporary directory for demo\ndemo_eml_dir = \"./demo_emails\"\nPath(demo_eml_dir).mkdir(exist_ok=True)\n\n# Write .eml files to disk\nwith open(f\"{demo_eml_dir}/email_1.eml\", \"w\") as f:\n    f.write(sample_eml_content_1)\n    print(f\"✅ Created: {demo_eml_dir}/email_1.eml\")\n\nwith open(f\"{demo_eml_dir}/email_2.eml\", \"w\") as f:\n    f.write(sample_eml_content_2)\n    print(f\"✅ Created: {demo_eml_dir}/email_2.eml\")\n\nprint(f\"\\n📁 Demo directory: {demo_eml_dir}/\")\nprint(\"\\n🎯 These files are now ready for parsing!\")"

In [ ]:
# 🚀 PARSE .eml FILES AND CONVERT TO JSON\n# ==========================================\n# This demonstrates the real workflow:\n# Gmail Exports (.eml files) → Parsed JSON → AI Assistant → Professional Replies\n\nprint(\"🚀 PARSING .eml FILES - DEMONSTRATION\\n\")\nprint(\"Step 1: Load .eml files from directory\")\nprint(\"=\" * 100)\n\n# Parse all .eml files from our demo directory\nparsed_emails_from_eml = parse_eml_directory(demo_eml_dir)\nprint(f\"\\n✅ Successfully parsed {len(parsed_emails_from_eml)} emails from .eml files\\n\")\n\nprint(\"Step 2: Display parsed email data\")\nprint(\"=\" * 100)\n\n# Display detailed information about each parsed email\nfor i, email in enumerate(parsed_emails_from_eml, 1):\n    print(f\"\\n📧 EMAIL {i} (From .eml file)\")\n    print(\"-\" * 100)\n    print(f\"  📧 Sender Name: {email['sender_name']}\")\n    print(f\"  📨 Sender Email: {email['sender']}\")\n    print(f\"  📌 Subject: {email['subject']}\")\n    print(f\"  💬 Message Preview (first 80 chars): {email['body'][:80]}...\")\n    print(f\"  🏷️  Type: {email['email_type']}\")\n    print(f\"  ⚡ Priority: {email['priority']}\")\n\nprint(\"\\n\" + \"=\" * 100)\nprint(\"✨ TRANSFORMATION COMPLETE!\")\nprint(f\"\\n📊 Summary:\")\nprint(f\"   • Input: .eml files (binary email format)\")\nprint(f\"   • Output: Structured JSON data\")\nprint(f\"   • Ready for: AI Email Assistant processing\")\nprint(f\"\\n💡 Next Step: Feed these emails to our AI assistant functions!\")\nprint(f\"   • generate_email_reply(parsed_emails_from_eml[0])\")\nprint(f\"   • generate_personalized_reply(parsed_emails_from_eml[1], tone='friendly')\")\nprint(f\"   • generate_reply_with_memory(parsed_emails_from_eml[0], thread_id='ticket_001')\")"

In [ ]:
# 🎯 COMPLETE PRODUCTION WORKFLOW DEMONSTRATION\n# ===============================================\n# Real-World Scenario:\n# You export emails from Gmail → Parse them → Generate professional replies\n\nprint(\"\\n\" + \"=\"*100)\nprint(\"🎯 COMPLETE PRODUCTION WORKFLOW: .eml FILES → AI EMAIL ASSISTANT\")\nprint(\"=\"*100 + \"\\n\")\n\n# STEP 1: Parse emails from .eml directory (simulating Gmail export)\nprint(\"📂 STEP 1: Loading emails from .eml files...\")\nprint(\"-\" * 100)\nproduction_emails = parse_eml_directory(demo_eml_dir)\nprint(f\"✅ Loaded {len(production_emails)} emails\")\nfor i, email in enumerate(production_emails, 1):\n    print(f\"   [{i}] From {email['sender_name']}: {email['subject'][:50]}...\")\n\n# STEP 2: Generate AI replies for each email\nprint(f\"\\n🤖 STEP 2: Generating AI replies using LangChain + OpenAI...\")\nprint(\"-\" * 100)\n\ngenerated_production_replies = []\nfor email in production_emails:\n    thread_id = f\"email_{email['id']}\"\n    \n    print(f\"\\n⚙️  Processing: {email['sender_name']} - {email['subject'][:50]}...\")\n    \n    # Generate reply with memory (most advanced version)\n    reply = generate_reply_with_memory(\n        email,\n        thread_id=thread_id,\n        your_name=\"Support Team\",\n        your_title=\"Customer Success Manager\",\n        company=\"Tech Solutions Inc.\",\n        tone=\"professional\"\n    )\n    \n    generated_production_replies.append(reply)\n    \n    if reply['status'] == 'generated':\n        print(f\"   ✅ Reply generated successfully\")\n        print(f\"   📝 Subject: {reply['subject']}\")\n        print(f\"   💬 Preview: {reply['reply_body'][:100]}...\")\n    else:\n        print(f\"   ❌ Failed: {reply.get('error')}\")\n\n# STEP 3: Summary\nprint(f\"\\n\" + \"=\"*100)\nprint(\"✨ WORKFLOW COMPLETE!\")\nprint(\"=\"*100)\n\nprint(\"\\n📊 Results:\")\nprint(f\"   ✅ Emails processed: {len(generated_production_replies)}\")\nprint(f\"   ✅ Successful replies: {sum(1 for r in generated_production_replies if r['status'] == 'generated')}\")\nprint(f\"   ❌ Failed: {sum(1 for r in generated_production_replies if r['status'] == 'failed')}\")\n\nprint(\"\\n🎓 WHAT YOU'VE LEARNED:\")\nprint(\"   1. ✅ Emails exist as .eml files when exported from email clients\")\nprint(\"   2. ✅ Parse them using Python's email.parser.BytesParser\")\nprint(\"   3. ✅ Convert to JSON format for AI processing\")\nprint(\"   4. ✅ Feed directly into LangChain Email Assistant\")\nprint(\"   5. ✅ Maintain conversation context with thread memory\")\n\nprint(\"\\n🚀 DEPLOYMENT ROADMAP FOR PRODUCTION:\")\nprint(\"   ➡️  Step 1: Export your Gmail/Outlook folder as .eml files\")\nprint(\"   ➡️  Step 2: Point parse_eml_directory() to that folder\")\nprint(\"   ➡️  Step 3: Create API endpoint for automation (FastAPI/Flask)\")\nprint(\"   ➡️  Step 4: Add database for persistent email history (PostgreSQL/MongoDB)\")\nprint(\"   ➡️  Step 5: Deploy to cloud (AWS Lambda, Google Cloud Functions, etc.)\")\nprint(\"   ➡️  Step 6: Set up email scheduling and webhooks\")\nprint(\"   ➡️  Step 7: Monitor performance and fine-tune prompts\")\nprint(\"\\n💼 Business Impact:\")\nprint(\"   • 📈 10x faster email responses\")\nprint(\"   • 💰 Reduce human support workload by 60%\")\nprint(\"   • ⏰ Handle 24/7 customer inquiries\")\nprint(\"   • 📊 Consistent, professional communication\")"

## 🎓 CHALLENGE EXERCISE: Parse Your Own Email!

**Objective:** Create your own .eml file and parse it using the functions you just learned.

### Steps:
1. **Create an .eml file** using the template below
2. **Save it** to the `./demo_emails/` directory
3. **Parse it** using `parse_eml_file()`
4. **Verify the output** matches your expectations

### .eml File Template:
```
From: your.email@example.com
To: recipient@company.com
Subject: Your custom email subject
Date: Mon, 15 Jan 2024 10:30:00 +0000
Content-Type: text/plain; charset="UTF-8"

Write your email body here. You can make it as long and detailed as you want.
This template works with the BytesParser function we created.

Feel free to experiment with different:
- Subject lines
- Email content
- Sender names and addresses
```

### Code Template:
```python
# 1. Create your email file
eml_content = """From: your.email@example.com
To: recipient@company.com
Subject: Your subject here
Date: Mon, 15 Jan 2024 10:30:00 +0000
Content-Type: text/plain; charset="UTF-8"

Your email body here!
"""

# 2. Save it
with open("./demo_emails/my_test_email.eml", "w") as f:
    f.write(eml_content)

# 3. Parse it
my_email = parse_eml_file("./demo_emails/my_test_email.eml", 
                          email_id="challenge_1",
                          email_type="support",
                          priority="high")

# 4. Display the result
print(f"Sender: {my_email['sender_name']}")
print(f"Subject: {my_email['subject']}")
print(f"Body: {my_email['body'][:200]}...")

# 5. Generate a reply!
reply = generate_email_reply(my_email)
print(f"\n🤖 AI Reply:\n{reply}")
```

**What to Try Next:**
- ✨ Experiment with multipart MIME emails (include HTML + Plain Text)
- ✨ Try parsing an email with attachments (challenge: handle the file paths)
- ✨ Create a batch of 5 emails and generate personalized replies for each
- ✨ Store the AI-generated replies back to .eml files

---

## ⚠️ Important Notes for Students

### Before Running This Notebook:

1. **API Key Setup** ✅
   - Create account at https://platform.openai.com/
   - Generate API key from Settings → API Keys
   - Create `.env` file in same directory as this notebook
   - Add: `OPENAI_API_KEY=sk-...`
   - Never commit `.env` to Git!

2. **Library Installation** ✅
   ```bash
   pip install langchain langchain-openai python-dotenv pandas
   ```

3. **Cost Warning** 💰
   - OpenAI charges per API call
   - Each email costs ~$0.001-0.01 depending on length
   - Starting credit: $5 (usually lasts a long time)
   - Monitor usage at: https://platform.openai.com/usage

4. **Rate Limits** 🚦
   - Free tier: 3 requests/minute
   - This is fine for learning, but not production
   - Upgrade plan for higher limits

### Common Issues & Solutions:

| Issue | Solution |
|-------|----------|
| `ModuleNotFoundError: No module named 'langchain'` | Run: `pip install langchain langchain-openai` |
| `Invalid API key` | Check your `.env` file has correct OPENAI_API_KEY |
| `RateLimitError` | Wait a minute and try again, or upgrade your OpenAI plan |
| `Email body is empty` | Some .eml files may have HTML only; our parser looks for plain text |
| `Thread memory not working` | Make sure you use the same `thread_id` for related emails |

### Customization Ideas:

**Change Default Names/Company:**
```python
# Instead of this:
reply = generate_email_reply(email)

# Do this:
reply = generate_email_reply(
    email,
    your_name="Your Name",
    your_title="Your Title",
    company="Your Company"
)
```

**Use Different LLM Models:**
```python
# Instead of gpt-4o-mini:
llm = ChatOpenAI(
    model="gpt-4",      # More capable but more expensive
    temperature=0.5,    # Lower = more consistent
    max_tokens=1000     # Longer responses
)
```

**Export More Data to CSV:**
```python
# Customize what gets saved
df_replies.to_csv('detailed_replies.csv', index=False)
df_replies[['to', 'subject', 'generated_at']].to_csv('summary.csv')
```

---

## 🎓 Learning Tips

✅ **Do:**
- Run each cell one at a time and read the output
- Modify sample emails to see different responses
- Try different tones and compare results
- Create your own email examples
- Save outputs and review quality

❌ **Don't:**
- Run all cells at once (hard to debug)
- Ignore error messages (they're helpful!)
- Skip the explanations (understand the "why")
- Use this for spam or harmful purposes
- Share your API key

---

## 📊 Success Criteria

By the end of this notebook, you should be able to:

✓ Explain how LangChain chains work (Prompt → LLM)
✓ Use ChatOpenAI to generate professional emails
✓ Create and customize prompt templates
✓ Understand conversation memory and threads
✓ Process real emails from Gmail/Outlook
✓ Generate replies in different tones
✓ Build a production-ready email system
✓ Debug and troubleshoot common issues

---

## 🔧 Debugging & Troubleshooting Guide

### Problem: "ModuleNotFoundError: No module named 'langchain'"

**Cause:** Library not installed
**Solution:**
```bash
pip install langchain langchain-openai python-dotenv pandas
```

---

### Problem: "Invalid API key provided"

**Cause:** OPENAI_API_KEY not found or incorrect
**Solution:**
1. Create `.env` file in same folder as notebook
2. Add: `OPENAI_API_KEY=sk-xxxxxxxxxxxx`
3. Make sure no extra spaces
4. Reload the notebook: Kernel → Restart
5. Verify with: `print(os.getenv("OPENAI_API_KEY"))`

---

### Problem: "RateLimitError: Rate limit exceeded"

**Cause:** Too many API calls too quickly
**Solution:**
1. Wait 1 minute before running again
2. Reduce batch size
3. Upgrade your OpenAI plan

---

### Problem: "No previous exchange(s) in memory"

**Cause:** Thread memory not working
**Solution:**
1. Check `thread_id` is consistent across emails
2. Verify with: `history = get_session_history('thread-id'); print(len(history.messages))`

---

## 💡 Pro Tips

- **Always run STEP 1 first** to load libraries
- **Test with sample_emails before using your data**
- **Save successful replies** to backup folder
- **Monitor your API usage** at platform.openai.com
- **Add error handling** to production code